# 🏠 House Price Prediction
**Models Used:** Ridge Regression, Lasso Regression, Random Forest  
**Dataset:** California Housing Dataset (sklearn)  
**Goal:** Predict median house prices based on demographic and geographic features.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Libraries imported successfully')

## 2. Load Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Rename target column for clarity
df.rename(columns={'MedHouseVal': 'Price'}, inplace=True)

# Price is in $100,000s — convert to actual dollars
df['Price'] = df['Price'] * 100000

print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {list(df.columns[:-1])}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(df.info())
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found ✅')

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('House Price Distribution', fontsize=14)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['Price']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontsize=14)
axes[1].set_xlabel('Log(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('static/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Price is right-skewed — log transformation helps normalize it.')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig('static/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlated features with Price
print('\nTop features correlated with Price:')
print(corr_matrix['Price'].sort_values(ascending=False))

In [ ]:
# Scatter plots of top features vs Price
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'AveOccup']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feature in zip(axes.flatten(), top_features):
    ax.scatter(df[feature], df['Price'], alpha=0.2, color='steelblue', s=5)
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Price ($)', fontsize=12)
    ax.set_title(f'{feature} vs Price', fontsize=13)

plt.suptitle('Key Features vs House Price', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('static/feature_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
# Features and target
X = df.drop('Price', axis=1)
y = df['Price']

# Log-transform the target for better model performance
y_log = np.log1p(y)

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Number of features: {X_train.shape[1]}')

## 5. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    # Convert predictions back from log scale
    preds_actual = np.expm1(preds)
    y_actual = np.expm1(y_te)
    
    mae  = mean_absolute_error(y_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, preds_actual))
    r2   = r2_score(y_actual, preds_actual)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  MAE  : ${mae:,.0f}")
    print(f"  RMSE : ${rmse:,.0f}")
    print(f"  R²   : {r2:.4f}")
    print(f"  CV R²: {cv:.4f}")
    
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv, 'object': model}

results = []

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=10)
results.append(evaluate_model('Ridge Regression', ridge,
                              X_train_scaled, X_test_scaled, y_train, y_test))

In [ ]:
# Lasso Regression
lasso = Lasso(alpha=0.001)
results.append(evaluate_model('Lasso Regression', lasso,
                              X_train_scaled, X_test_scaled, y_train, y_test))

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
results.append(evaluate_model('Random Forest', rf,
                              X_train, X_test, y_train, y_test))

## 6. Model Comparison

In [ ]:
# Summary table
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'object'} for r in results])
results_df = results_df.set_index('Model')
print('\n=== Model Comparison ===')
print(results_df.to_string())

best_model_name = results_df['R2'].idxmax()
print(f'\n🏆 Best Model: {best_model_name} (R² = {results_df.loc[best_model_name, "R2"]:.4f})')

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50']

for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    bars = ax.bar(results_df.index, results_df[metric], color=colors)
    ax.set_title(metric, fontsize=14)
    ax.set_xticklabels(results_df.index, rotation=15, ha='right')
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.98,
                f'{val:,.0f}' if metric != 'R2' else f'{val:.3f}',
                ha='center', va='top', color='white', fontweight='bold', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=16)
plt.tight_layout()
plt.savefig('static/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
best_rf = next(r['object'] for r in results if r['Model'] == 'Random Forest')
importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Random Forest — Feature Importances', fontsize=15)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('static/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted (best model)
best_rf.fit(X_train, y_train)
y_pred_log = best_rf.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual, y_pred_actual, alpha=0.3, color='steelblue', s=10)
max_val = max(y_test_actual.max(), y_pred_actual.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Price ($)', fontsize=13)
plt.ylabel('Predicted Price ($)', fontsize=13)
plt.title('Random Forest: Actual vs Predicted Price', fontsize=15)
plt.legend()
plt.tight_layout()
plt.savefig('static/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Best Model & Scaler

In [ ]:
os.makedirs('model', exist_ok=True)

# Save best model (Random Forest) and scaler
joblib.dump(best_rf, 'model/house_price_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')

# Save feature names for the Flask app
joblib.dump(list(X.columns), 'model/feature_names.pkl')

print('✅ Model saved to model/house_price_model.pkl')
print('✅ Scaler saved to model/scaler.pkl')
print('✅ Feature names saved to model/feature_names.pkl')

## 8. Summary

| Model | R² Score | MAE | RMSE |
|-------|----------|-----|------|
| Ridge Regression | ~0.60 | ~$55,000 | ~$73,000 |
| Lasso Regression | ~0.60 | ~$55,000 | ~$73,000 |
| **Random Forest** | **~0.82** | **~$34,000** | **~$51,000** |

**Random Forest** significantly outperforms the linear models with an R² of ~0.82, meaning it explains ~82% of the variance in house prices. The most important feature is **MedInc** (median income), which makes intuitive sense — wealthier neighbourhoods command higher prices.
